# 深度學習框架應用（大學部）— 第 4 週：全連接神經網路 MLP

- **來源投影片**：`dlu_w04.json`（第 4 週，共 4 張程式投影片 → 本筆記本 5 個程式儲存格，含「環境準備」1 格）
- **上課方式**：本課程於 Google Colab 上機（PyTorch 為主）。建議先於「執行階段 → 變更執行階段類型」選擇 GPU。
- **共同注意事項**：程式內容一字不改照抄自投影片；執行必要但投影片未含的膠水碼集中在「環境準備」格（或以行內註解標明），一律標註 `# ←投影片未含，執行所需`。
- **本週注意事項**：
  - 本週使用 torchvision 內建 MNIST，第一次執行會自動下載，無須準備資料。

## 環境準備

下方儲存格為**投影片未含**、但執行本週程式所需的補充（import、安裝、資料路徑變數等），每行皆有標註。

In [ ]:
# ===== 環境準備 =====
# 本週使用 torchvision 內建 MNIST（第一次執行會自動下載），毋須額外設定。
import torch  # ←投影片未含，執行所需（僅作環境確認）
print('CUDA 可用：', torch.cuda.is_available())

### 任務一：載入 MNIST
**投影片儲存格標示**：`cell 1 — data.py`｜LAB TASK 1

> **投影片說明**：0.1307 與 0.3081 是 MNIST 訓練集的平均與標準差。訓練集 60,000 張切成 54,000／6,000，測試集 10,000 張全程不動，最後才用。

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import v2
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tf = v2.Compose([v2.ToImage(),
                 v2.ToDtype(torch.float32, scale=True),
                 v2.Normalize((0.1307,), (0.3081,))])
train_full = datasets.MNIST('./data', train=True, download=True,
                            transform=tf)
test_ds = datasets.MNIST('./data', train=False, download=True,
                         transform=tf)
g = torch.Generator().manual_seed(42)
train_ds, val_ds = random_split(train_full, [54000, 6000], g)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=256)
test_dl  = DataLoader(test_ds,  batch_size=256)

### 任務二：定義 MLP 模型
**投影片儲存格標示**：`cell 2 — model.py`｜LAB TASK 2

> **投影片說明**：最後一層不接激活也不接 Dropout，直接輸出 10 維 logits，交給 CrossEntropyLoss 處理。這個設定印出來的參數量是 235,914，包含兩層 BatchNorm 各自的縮放與位移參數，可以自己動手驗算一次。

In [ ]:
from torch import nn
class MLP(nn.Module):
    def __init__(self, in_dim=784, hid=(256,128), ncls=10, p=0.2):
        super().__init__()             # 一定要先呼叫父類別建構式
        layers, prev = [], in_dim
        for h in hid:
            layers += [nn.Linear(prev, h),
                       nn.BatchNorm1d(h),   # 線性層後、激活前
                       nn.ReLU(),
                       nn.Dropout(p)]       # 激活之後
            prev = h
        layers.append(nn.Linear(prev, ncls))  # 輸出 logits
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        x = x.flatten(1)               # (B,1,28,28) -> (B,784)
        return self.net(x)
model = MLP().to(dev)
print(sum(p.numel() for p in model.parameters()))

### 任務三：共用的 run_epoch 函式
**投影片儲存格標示**：`cell 3 — train.py`｜LAB TASK 3

> **投影片說明**：把訓練與驗證寫成同一個函式，用旗標切換 model.train()／model.eval() 與是否建圖，可以避免兩份程式各改一半而不一致 —— 這是實務上很常見的 bug 來源。

In [ ]:
import torch
crit = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(dl, train=True):
    model.train() if train else model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):   # 驗證時自動不建圖
        for x, y in dl:
            x, y = x.to(dev), y.to(dev)
            out = model(x)
            loss = crit(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot_loss += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += y.size(0)
    return tot_loss / n, correct / n

### 任務四：訓練、保存與最終評估
**投影片儲存格標示**：`cell 4 — fit.py`｜LAB TASK 4

> **投影片備註**：保存的是 state_dict（純權重字典），不是整個模型物件 —— 這樣換環境載入才不會因為類別路徑改變而失敗。
> **投影片說明**：把每個 epoch 的四個數字另外存進串列，就能用 matplotlib 畫出訓練曲線 —— 這張圖是本週作業的必繳項目。

In [ ]:
best = 0.0
for ep in range(1, 21):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    va_loss, va_acc = run_epoch(val_dl,   train=False)
    print('ep%02d  train %.4f/%.4f  val %.4f/%.4f'
          % (ep, tr_loss, tr_acc, va_loss, va_acc))
    if va_acc > best:            # 只保留驗證表現最好的權重
        best = va_acc
        torch.save(model.state_dict(), 'mlp_best.pt')

# 最終評估：載回最佳權重，測試集只跑這一次
model.load_state_dict(torch.load('mlp_best.pt', map_location=dev))
te_loss, te_acc = run_epoch(test_dl, train=False)
print('best val %.4f  test %.4f' % (best, te_acc))